In [9]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
Stratifiziertes Sampling aus NDJSON:
- Möglichst gleichmäßig verteilt über (best_topic_index, depth_bin)
- Zielgröße ~400
- Filtert removed/deleted/leere bodies raus
- Output-CSV enthält: comment_id, best_topic_index, body, predecessor
  wobei predecessor = Body des direkten Vorgänger-Kommentars (parent_id -> comment_id)
"""

from __future__ import annotations

import argparse
from pathlib import Path
import numpy as np
import pandas as pd


# ----------------------------
# Normalizer / Binner
# ----------------------------

def normalize_topic(series: pd.Series) -> pd.Series:
    """best_topic_index robust normalisieren (int wenn möglich, sonst str, NaN -> 'UNKNOWN')."""
    def norm(x):
        if pd.isna(x):
            return "UNKNOWN"
        if isinstance(x, (int, np.integer)):
            return int(x)
        if isinstance(x, (float, np.floating)) and float(x).is_integer():
            return int(x)
        return str(x)
    return series.map(norm)


def normalize_depth(series: pd.Series) -> pd.Series:
    """depth robust zu int normalisieren; nicht parsebare Werte -> NaN."""
    def norm(x):
        if pd.isna(x):
            return np.nan
        if isinstance(x, (int, np.integer)):
            return int(x)
        if isinstance(x, (float, np.floating)):
            if float(x).is_integer():
                return int(x)
            return np.nan
        if isinstance(x, str):
            t = x.strip()
            if t == "":
                return np.nan
            try:
                v = float(t)
                if v.is_integer():
                    return int(v)
                return np.nan
            except Exception:
                return np.nan
        return np.nan

    out = series.map(norm)
    out = out.where(out >= 0, np.nan)  # negative depth als ungültig behandeln
    return out


def make_depth_bins(depth: pd.Series, method: str = "quantile", bins: int = 10) -> pd.Series:
    """
    Erzeugt depth_bin für Stratifizierung.
    - quantile: ungefähr gleich viele Samples pro Bin
    - log: Buckets nach log2(1+depth)
    - raw: depth direkt verwenden (kann zu extrem vielen Strata führen)
    """
    method = method.lower()

    if method == "raw":
        return depth.map(lambda x: "UNKNOWN" if pd.isna(x) else int(x))

    if method == "log":
        def to_bucket(x):
            if pd.isna(x):
                return "UNKNOWN"
            b = int(np.floor(np.log2(1 + int(x))))
            return f"log2_1p_depth={b}"
        return depth.map(to_bucket)

    if method == "quantile":
        d = depth.copy()
        mask = d.notna()
        if mask.sum() == 0:
            return pd.Series(["UNKNOWN"] * len(d), index=d.index)

        q = pd.qcut(d[mask], q=bins, duplicates="drop")
        labels = q.astype(str)

        out = pd.Series(["UNKNOWN"] * len(d), index=d.index, dtype=object)
        out.loc[mask] = labels.values
        return out

    raise ValueError(f"Unknown depth binning method: {method}. Use quantile|log|raw.")


def is_valid_body(s: pd.Series) -> pd.Series:
    """
    True, wenn body 'inhaltlich' ist.
    Filtert: '', removed, [removed], deleted, [deleted] (case-insensitive, whitespace-robust)
    """
    clean = s.astype(str).str.strip().str.lower()
    bad = {"", "removed", "[removed]", "deleted", "[deleted]"}
    return ~clean.isin(bad)


# ----------------------------
# Sampling
# ----------------------------

def stratified_sample_equal(df: pd.DataFrame, n_total: int, key_cols: list[str], seed: int = 42) -> pd.DataFrame:
    """
    Gleichmäßig über Strata (key_cols).
    Vorgehen:
      1) Strata bilden
      2) Grundquote = floor(n_total / #strata)
      3) pro Stratum min(Grundquote, verfügbare Anzahl) ziehen
      4) Rest über Strata mit verbleibenden Einträgen verteilen (round-robin)
    """
    rng = np.random.default_rng(seed)
    groups = df.groupby(key_cols, dropna=False)

    strata = [(key, g.index.to_numpy()) for key, g in groups]
    if not strata:
        return df.iloc[0:0].copy()

    n_strata = len(strata)
    base = n_total // n_strata
    remainder = n_total - base * n_strata

    selected = []
    leftover = []

    for key, idx in strata:
        idx_shuf = idx.copy()
        rng.shuffle(idx_shuf)

        take = min(base, len(idx_shuf))
        if take > 0:
            selected.append(idx_shuf[:take])

        rem = idx_shuf[take:]
        if len(rem) > 0:
            leftover.append((key, rem))

    rng.shuffle(leftover)
    i = 0
    while remainder > 0 and leftover:
        key, rem = leftover[i]
        selected.append(rem[:1])
        rem2 = rem[1:]
        leftover[i] = (key, rem2)
        remainder -= 1

        if len(rem2) == 0:
            leftover.pop(i)
            if not leftover:
                break
            i %= len(leftover)
        else:
            i = (i + 1) % len(leftover)

    if not selected:
        return df.iloc[0:0].copy()

    chosen_idx = np.concatenate(selected)
    out = df.loc[chosen_idx].copy()
    out = out.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    return out


# ----------------------------
# Main
# ----------------------------

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--infile", type=str, default="data/conversation_threads_flat.ndjson",
                    help="Pfad zur NDJSON (lines=true).")
    ap.add_argument("--outfile", type=str, default="sampled_threads_400.xlsx",
                    help="Ziel-xlsx.")
    ap.add_argument("--n", type=int, default=400, help="Zielanzahl Einträge.")
    ap.add_argument("--seed", type=int, default=42, help="Random seed.")

    ap.add_argument("--depth-binning", type=str, default="quantile",
                    choices=["quantile", "log", "raw"],
                    help="Wie depth für Stratifizierung gebinnt wird.")
    ap.add_argument("--depth-bins", type=int, default=10,
                    help="Anzahl Bins für quantile-Binning (ignoriert bei log/raw).")

    ap.add_argument("--flatten-body", action="store_true",
                    help="Wenn gesetzt: ersetzt Zeilenumbrüche im body durch Spaces (Excel-freundlicher).")

    args, _ = ap.parse_known_args()

    infile = Path(args.infile)
    if not infile.exists():
        raise FileNotFoundError(f"Input-Datei nicht gefunden: {infile.resolve()}")

    df = pd.read_json(infile, lines=True)

    required = {"comment_id", "best_topic_index", "body", "depth", "parent_id"}
    missing = sorted(required - set(df.columns))
    if missing:
        raise KeyError(f"Diese Pflichtspalten fehlen im Datensatz: {missing}")

    # --- Für predecessor-lookup: body map bauen (vor Filtern fürs Sampling!)
    # Wir bereinigen nur "bad bodies" zu NaN/"" damit predecessor nicht removed/deleted zurückgibt.
    df["_body_valid"] = is_valid_body(df["body"])
    body_lookup = (
        df.loc[df["_body_valid"], ["comment_id", "body"]]
        .dropna(subset=["comment_id", "body"])
        .drop_duplicates(subset=["comment_id"], keep="last")
        .set_index("comment_id")["body"]
    )

    # --- Normalisieren fürs Sampling
    df["best_topic_index"] = normalize_topic(df["best_topic_index"])
    df["depth_norm"] = normalize_depth(df["depth"])
    df["depth_bin"] = make_depth_bins(df["depth_norm"], method=args.depth_binning, bins=args.depth_bins)

    # --- Filter: nur Zeilen mit sinnvollem body & comment_id
    df = df.dropna(subset=["comment_id", "body"]).copy()
    df = df[is_valid_body(df["body"])].copy()

    # --- Sampling
    n_target = min(args.n, len(df))
    key_cols = ["best_topic_index", "depth_bin"]
    sampled = stratified_sample_equal(df, n_total=n_target, key_cols=key_cols, seed=args.seed)

    # --- predecessor-Spalte: parent_id -> predecessor body
    # Falls parent_id Formate wie "t1_xxx" enthält, und comment_id auch "t1_xxx",
    # klappt das direkt. Wenn nicht, musst du ggf. ein Prefix normalisieren.
    sampled["predecessor"] = sampled["parent_id"].map(body_lookup).fillna("")

    # Optional: Excel-freundlicher (keine Zeilenumbrüche in Zellen)
    if args.flatten_body:
        sampled["body"] = sampled["body"].astype(str).str.replace(r"\s*\n\s*", " ", regex=True).str.strip()
        sampled["predecessor"] = sampled["predecessor"].astype(str).str.replace(r"\s*\n\s*", " ", regex=True).str.strip()

    # --- Output
    out = sampled[["comment_id", "best_topic_index", "body", "predecessor"]].copy()

    outfile = Path(args.outfile)
    outfile.parent.mkdir(parents=True, exist_ok=True)

    # Excel/DE-kompatibel: Semikolon + UTF-8 BOM
    out.to_excel(outfile.with_suffix(".xlsx"), index=False)

    # --- Report
    dist = sampled.groupby(key_cols).size().sort_values(ascending=False)
    print(f"Wrote {len(out)} rows to: {outfile.resolve()}")
    print(f"Strata count: {dist.shape[0]}")
    print("Top strata sizes:")
    print(dist.head(20).to_string())


if __name__ == "__main__":
    main()

Wrote 400 rows to: /Users/arthur/DataspellProjects/reddit-l/sampled_threads_400.xlsx
Strata count: 114
Top strata sizes:
best_topic_index  depth_bin    
0                 (-0.001, 1.0]    4
6                 (6.0, 86.0]      4
11                (4.0, 6.0]       4
                  (1.0, 2.0]       4
10                (6.0, 86.0]      4
                  (3.0, 4.0]       4
                  (2.0, 3.0]       4
                  (-0.001, 1.0]    4
9                 (6.0, 86.0]      4
                  (4.0, 6.0]       4
                  (2.0, 3.0]       4
                  (1.0, 2.0]       4
8                 (3.0, 4.0]       4
                  (2.0, 3.0]       4
                  (-0.001, 1.0]    4
7                 (6.0, 86.0]      4
                  (4.0, 6.0]       4
                  (3.0, 4.0]       4
                  (1.0, 2.0]       4
12                (1.0, 2.0]       4
